# 01 — Data Cleaning
ทำความสะอาดข้อมูลทั้ง 4 ชุดจาก `../data/` แล้วบันทึกผลลัพธ์ที่สะอาดไว้ที่ `../data/clean/`

| ชุดข้อมูล | ปัญหาที่พบ |
|-----------|------------|
| pharma-sales (daily/hourly/weekly/monthly) | สะอาด — แค่แปลง `datum` เป็น datetime + ตรวจช่วงเวลาขาดหาย |
| inventory (data.csv) | คอลัมน์วันหมดอายุซ้ำ 2 อัน, ค่าว่าง, คอลัมน์ที่ไม่ใช้ |
| supply-chain | 5 แถวซ้ำ |
| pharmacy-products | 3,911 ค่าว่าง, 10 แถวซ้ำ, `discount_percentage` เป็น text |

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)


def to_snake(name: str) -> str:
    """'Weekday Name' -> 'weekday_name', 'drugName' -> 'drug_name'."""
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name.strip())
    name = re.sub(r"[\s\-]+", "_", name)
    return name.lower()


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [to_snake(c) for c in df.columns]
    return df


def save(df: pd.DataFrame, name: str) -> None:
    out = CLEAN_DIR / f"{name}.csv"
    df.to_csv(out, index=False)
    print(f"[saved] {out.relative_to(DATA_DIR)}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")

## 1) pharma-sales — time series หลักสำหรับพยากรณ์
คอลัมน์ยาเป็นรหัสกลุ่ม ATC เก็บ mapping ชื่อไว้อ้างอิง

In [2]:
# รหัสกลุ่ม ATC: คำอธิบายอังกฤษ + ไทย
ATC_GROUPS = {
    "M01AB": ("Anti-inflammatory/antirheumatic - Acetic acid derivatives",
              "ยาต้านการอักเสบ/รูมาติก กลุ่มอนุพันธ์กรดอะซิติก"),
    "M01AE": ("Anti-inflammatory/antirheumatic - Propionic acid derivatives",
              "ยาต้านการอักเสบ/รูมาติก กลุ่มอนุพันธ์กรดโพรพิโอนิก (เช่น ไอบูโพรเฟน)"),
    "N02BA": ("Analgesics/antipyretics - Salicylic acid derivatives",
              "ยาแก้ปวด/ลดไข้ กลุ่มอนุพันธ์กรดซาลิไซลิก (เช่น แอสไพริน)"),
    "N02BE": ("Analgesics/antipyretics - Pyrazolones & Anilides",
              "ยาแก้ปวด/ลดไข้ กลุ่มไพราโซโลนและอะนิไลด์ (เช่น พาราเซตามอล)"),
    "N05B":  ("Psycholeptics - Anxiolytics",
              "ยาคลายกังวล"),
    "N05C":  ("Psycholeptics - Hypnotics & sedatives",
              "ยานอนหลับและยากล่อมประสาท"),
    "R03":   ("Drugs for obstructive airway diseases",
              "ยารักษาโรคทางเดินหายใจอุดกั้น (เช่น หอบหืด)"),
    "R06":   ("Antihistamines for systemic use",
              "ยาแก้แพ้ (แอนติฮิสตามีน) ชนิดออกฤทธิ์ทั่วร่างกาย"),
}
DRUG_COLS = list(ATC_GROUPS.keys())

atc_df = pd.DataFrame(
    [(code, en, th) for code, (en, th) in ATC_GROUPS.items()],
    columns=["atc_code", "description", "description_th"],
)
save(atc_df, "atc_drug_groups")
atc_df

[saved] clean\atc_drug_groups.csv  (8 rows x 3 cols)


,atc_code,description,description_th
0,M01AB,Anti-inflammatory/antirheumatic - Acetic acid ...,ยาต้านการอักเสบ/รูมาติก กลุ่มอนุพันธ์กรดอะซิติก
1,M01AE,Anti-inflammatory/antirheumatic - Propionic ac...,ยาต้านการอักเสบ/รูมาติก กลุ่มอนุพันธ์กรดโพรพิโ...
2,N02BA,Analgesics/antipyretics - Salicylic acid deriv...,ยาแก้ปวด/ลดไข้ กลุ่มอนุพันธ์กรดซาลิไซลิก (เช่น...
3,N02BE,Analgesics/antipyretics - Pyrazolones & Anilides,ยาแก้ปวด/ลดไข้ กลุ่มไพราโซโลนและอะนิไลด์ (เช่น...
4,N05B,Psycholeptics - Anxiolytics,ยาคลายกังวล
5,N05C,Psycholeptics - Hypnotics & sedatives,ยานอนหลับและยากล่อมประสาท
6,R03,Drugs for obstructive airway diseases,ยารักษาโรคทางเดินหายใจอุดกั้น (เช่น หอบหืด)
7,R06,Antihistamines for systemic use,ยาแก้แพ้ (แอนติฮิสตามีน) ชนิดออกฤทธิ์ทั่วร่างกาย


In [3]:
SALES_FREQ = {
    "salesdaily": "D",
    "saleshourly": "h",
    "salesweekly": "W",    # ข้อมูลอยู่บนวันอาทิตย์ ตรงกับ W-SUN
    "salesmonthly": "ME",  # ข้อมูลเป็น "สิ้นเดือน" (เช่น 2014-01-31) จึงใช้ month-end
}

# รหัสยา ATC (M01AB ฯลฯ) ต้องคงตัวพิมพ์ใหญ่ไว้ จึง rename เฉพาะคอลัมน์ metadata
META_RENAME = {"Year": "year", "Month": "month", "Hour": "hour", "Weekday Name": "weekday_name"}

for fname, freq in SALES_FREQ.items():
    df = pd.read_csv(DATA_DIR / "pharma-sales-data" / f"{fname}.csv")
    df = df.rename(columns=META_RENAME)

    # datum -> datetime, ใช้เป็น index เวลา
    df["datum"] = pd.to_datetime(df["datum"], errors="coerce", dayfirst=False)
    df = df.dropna(subset=["datum"]).drop_duplicates(subset=["datum"])
    df = df.sort_values("datum").set_index("datum")

    # ตรวจช่วงเวลาที่ขาดหาย แล้ว reindex ให้ต่อเนื่อง (เติม 0 = ไม่มียอดขาย)
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq=freq)
    missing = len(full_idx) - len(df)
    df = df.reindex(full_idx)
    df[DRUG_COLS] = df[DRUG_COLS].fillna(0)
    df.index.name = "datum"

    print(f"{fname}: missing periods filled = {missing}")
    save(df.reset_index(), fname)

salesdaily: missing periods filled = 0
[saved] clean\salesdaily.csv  (2,106 rows x 13 cols)
saleshourly: missing periods filled = 0
[saved] clean\saleshourly.csv  (50,532 rows x 13 cols)
salesweekly: missing periods filled = 0
[saved] clean\salesweekly.csv  (302 rows x 9 cols)
salesmonthly: missing periods filled = 0
[saved] clean\salesmonthly.csv  (70 rows x 9 cols)


## 2) inventory — รวมคอลัมน์วันหมดอายุที่ซ้ำกัน + ตัดคอลัมน์ที่ไม่ใช้

In [4]:
inv = pd.read_csv(DATA_DIR / "inventory-data-for-pharmacy-website-in-json-format" / "data.csv")
inv = clean_columns(inv)

# ไฟล์มีคอลัมน์วันหมดอายุ 2 อัน: 'expirydate' และ 'expiryDate'
# (หลัง snake_case กลายเป็น 'expirydate' กับ 'expiry_date')
# -> รวมเป็นคอลัมน์เดียว โดยเติมค่าจากอีกอันที่ไม่ว่าง แล้วลบต้นฉบับทั้งคู่
date_cols = [c for c in inv.columns if c.startswith("expiry")]
merged = inv[date_cols[0]].copy()
for c in date_cols[1:]:
    merged = merged.fillna(inv[c])
inv = inv.drop(columns=date_cols)
inv["expiry_date"] = pd.to_datetime(merged, errors="coerce")

# ตัดคอลัมน์ที่ไม่ใช้ในโมเดล
drop_cols = ["image", "description", "disclaimer", "side_effects"]
inv = inv.drop(columns=[c for c in drop_cols if c in inv.columns])

# ลบแถวที่ไม่มีชื่อยา และแถวซ้ำ
inv = inv.dropna(subset=["drug_name"]).drop_duplicates()

print("คอลัมน์สุดท้าย:", list(inv.columns))
print("วันหมดอายุที่ parse ไม่ได้:", inv["expiry_date"].isnull().sum())
save(inv, "inventory")

คอลัมน์สุดท้าย: ['drug_name', 'manufacturer', 'consume_type', 'price', 'category', 'count_in_stock', 'expiry_date']
วันหมดอายุที่ parse ไม่ได้: 4
[saved] clean\inventory.csv  (93 rows x 7 cols)


## 3) supply-chain — ลบแถวซ้ำ

In [5]:
sc = pd.read_csv(
    DATA_DIR / "pharmaceutical-supply-chain-optimization" / "Pharmaceutical Supply Chain Optimization.csv"
)
sc = clean_columns(sc)

before = len(sc)
sc = sc.drop_duplicates().reset_index(drop=True)
print(f"ลบแถวซ้ำ: {before - len(sc)}")

# restocking_strategy เป็น categorical ที่มีลำดับ
sc["restocking_strategy"] = pd.Categorical(
    sc["restocking_strategy"], categories=["Weekly", "Monthly", "Quarterly"], ordered=True
)
print(sc["restocking_strategy"].value_counts())
save(sc, "supply_chain")

ลบแถวซ้ำ: 5
restocking_strategy
Quarterly    33522
Weekly       33436
Monthly      33037
Name: count, dtype: int64
[saved] clean\supply_chain.csv  (99,995 rows x 4 cols)


## 4) pharmacy-products — แก้ discount เป็นตัวเลข + จัดการค่าว่าง + แยก packaging

In [6]:
prod = pd.read_csv(DATA_DIR / "pharmacy-products-dataset" / "Pharmacy_Products.csv")
prod = clean_columns(prod)

# 1) ลบแถวซ้ำ
before = len(prod)
prod = prod.drop_duplicates().reset_index(drop=True)
print(f"ลบแถวซ้ำ: {before - len(prod)}")

# 2) discount_percentage: '15% discount' -> 15.0 ; ค่าว่าง = ไม่มีส่วนลด = 0
prod["discount_percentage"] = (
    prod["discount_percentage"].str.extract(r"(\d+(?:\.\d+)?)").astype(float).fillna(0)
)

# 3) ถ้าไม่มีราคาลด ให้ใช้ราคาเต็ม
prod["discounted_price"] = prod["discounted_price"].fillna(prod["price"])

# 4) แยกปริมาณ/หน่วยจาก packaging เช่น '15 tablets' -> qty=15, unit='tablets'
pack = prod["packaging"].str.extract(r"(?P<pack_qty>[\d.]+)\s*(?P<pack_unit>[a-zA-Z]+)")
prod["pack_qty"] = pd.to_numeric(pack["pack_qty"], errors="coerce")
prod["pack_unit"] = pack["pack_unit"].str.lower()

print("ค่าว่างคงเหลือ:", prod.isnull().sum().sum())
save(prod, "pharmacy_products")

ลบแถวซ้ำ: 10
ค่าว่างคงเหลือ: 0
[saved] clean\pharmacy_products.csv  (6,188 rows x 7 cols)


## สรุป — ไฟล์ที่สะอาดใน `../data/clean/`

In [7]:
for fp in sorted(CLEAN_DIR.glob("*.csv")):
    df = pd.read_csv(fp)
    print(f"{fp.name:30s} {df.shape[0]:>7,} rows x {df.shape[1]} cols")

atc_drug_groups.csv                  8 rows x 3 cols
inventory.csv                       93 rows x 7 cols
pharmacy_products.csv            6,188 rows x 7 cols
salesdaily.csv                   2,106 rows x 13 cols
saleshourly.csv                 50,532 rows x 13 cols
salesmonthly.csv                    70 rows x 9 cols
salesweekly.csv                    302 rows x 9 cols
supply_chain.csv                99,995 rows x 4 cols
